# AIDA 2156 Project Exercise 06 – Process Step Bottlenecks

**Question:** How long does each step of the synthetic onboarding workflow take, which step is the longest observed stage, and what explanations are worth investigating?

This notebook uses the original synthetic `requests.csv` from AIDA 1141 Exercise 02. One row represents one onboarding task. It splits the receipt-to-team-completion time from AIDA 2156 Exercise 03 into the information check, assignment, and team processing, and reports coordinator confirmation separately because it happens after the team finishes.

Step durations are elapsed working time. They can include waiting, so they do not measure active staff effort or prove that a team lacks capacity.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("../../../AIDA-1141/project-labs/02-synthetic-request-records/data/requests.csv")
EX03_SUMMARY = Path("../03-processing-time-distributions/outputs/duration_summary.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TIMESTAMPS = ["submitted_at", "info_complete_at", "assigned_at", "team_completed_at", "completion_confirmed_at"]
# Only truly empty cells become missing (NaN); the text "not_applicable" stays as a value
requests = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[""], parse_dates=TIMESTAMPS)
assert all(pd.api.types.is_datetime64_any_dtype(requests[c]) for c in TIMESTAMPS), \
    "A timestamp column was not parsed as datetimes."
print(f"Loaded {len(requests):,} requests")

Loaded 2,064 requests


## A. Timestamp checks

The working-time measure assumes every timestamp is present, in workflow order, and on a weekday between 08:00 and 16:30, the working day defined in the AIDA 1141 Exercise 02 data card. These checks confirm those assumptions before any step duration is calculated, as in Exercise 03.

In [2]:
DAY_START = pd.Timedelta(hours=8)                  # working day 08:00-16:30, Monday-Friday (AIDA 1141 Exercise 02 data card)
DAY_END = pd.Timedelta(hours=16, minutes=30)
WORKDAY_HOURS = (DAY_END - DAY_START) / pd.Timedelta(hours=1)  # 8.5

time_of_day = {c: requests[c] - requests[c].dt.normalize() for c in TIMESTAMPS}
checks = pd.Series({
    "Request IDs are complete and unique": requests["request_id"].notna().all() and requests["request_id"].is_unique,
    "No blank timestamps": not requests[TIMESTAMPS].isna().any().any(),
    "Timestamps in workflow order": all((requests[a] <= requests[b]).all() for a, b in zip(TIMESTAMPS, TIMESTAMPS[1:])),
    "All timestamps on weekdays": all((requests[c].dt.dayofweek < 5).all() for c in TIMESTAMPS),
    "All timestamps within 08:00-16:30": all(((t >= DAY_START) & (t <= DAY_END)).all() for t in time_of_day.values()),
}, name="passed")
assert checks.all(), "A timestamp check failed; the working-time measure would be unreliable."
checks.to_frame()

,passed
Request IDs are complete and unique,True
No blank timestamps,True
Timestamps in workflow order,True
All timestamps on weekdays,True
All timestamps within 08:00-16:30,True
